In [19]:
import torch
import torch.nn as nn
import time
from torch.utils.data import TensorDataset, DataLoader
from preprocess import get_data

# Add the TransformerModel import
from transformer_model import TransformerModel

In [5]:
!pip install pandas numpy scikit-learn matplotlib


  Using cached pandas-2.2.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2025.1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.1-cp313-cp313-win_amd64.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached pandas-2.2.3-cp313-cp313-win_amd64.whl (11.5 MB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -------------- ------------------------- 4.5/12.6 MB 21.9 MB/s eta 0:00:01
   ---------------------------------- ----- 10.7/12.6 MB 26.3 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 26.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   ------------------------ --------------- 6.8/11.1 MB 33.6 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.1 MB 27.6 MB/s eta 0:00:01
   ---------------------------------------- 11.1/11.1 MB 26.0 MB/s 

In [12]:
!pip install torch-models


  Using cached torch_models-0.0.7-py3-none-any.whl.metadata (861 bytes)
Using cached torch_models-0.0.7-py3-none-any.whl (6.1 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 14.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import time
from torch.utils.data import TensorDataset, DataLoader
from preprocess import get_data

In [12]:
# Load and preprocess data
data = get_data()
X_train_tensor = torch.tensor(data["X_train_scaled"], dtype=torch.float32)
y_train_tensor = torch.tensor(data["y_train"], dtype=torch.float32).unsqueeze(1)
X_val_tensor = torch.tensor(data["X_val_scaled"], dtype=torch.float32)
y_val_tensor = torch.tensor(data["y_val"], dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(data["X_test_scaled"], dtype=torch.float32)
y_test_tensor = torch.tensor(data["y_test"], dtype=torch.float32).unsqueeze(1)

Loading preprocessed data and scaler from disk...


In [8]:
data.keys()

dict_keys(['X_train_scaled', 'y_train', 'X_val_scaled', 'y_val', 'X_test_scaled', 'y_test', 'X_train_sliding_window_scaled', 'y_train_sliding_window', 'X_val_sliding_window_scaled', 'y_val_sliding_window', 'X_test_sliding_window_scaled', 'y_test_sliding_window'])

In [13]:
batch_size = 32
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size)




In [14]:
def train_model(
    model, train_loader, val_loader, num_epochs=10, lr=0.001, model_save_path=None
):
    start_time = time.time()
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Training on device: {device}")
    model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    best_epoch = -1

    for epoch in range(num_epochs):
        start_time_epoch = time.time()
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)

        # Validate the model
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                predictions = model(X_batch)
                loss = criterion(predictions, y_batch)
                val_loss += loss.item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)

        print(
            f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}"
        )

        # Save the model if validation loss is the best seen so far
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            if model_save_path is not None:
                torch.save(model.state_dict(), model_save_path)
                print(
                    f"Saved best model at epoch {epoch+1} with val loss {best_val_loss:.4f}"
                )

        print(f"Time taken for epoch {epoch+1}: {time.time() - start_time_epoch:.2f}s")

    # Optionally, load the best model state at the end of training
    if best_epoch != -1 and model_save_path is not None:
        model.load_state_dict(torch.load(model_save_path))
        print(
            f"Loaded best model from epoch {best_epoch+1} with val loss {best_val_loss:.4f}"
        )

    print(f"Total time taken: {time.time() - start_time:.2f}s")

    return model

In [15]:
def evaluate_model(model, test_loader):
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Evaluating on device: {device}")
    model.to(device)
    model.eval()
    criterion = nn.MSELoss()
    test_loss = 0.0
    total_abs_error = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            test_loss += loss.item() * X_batch.size(0)
            total_abs_error += torch.sum(torch.abs(predictions - y_batch)).item()
    test_loss /= len(test_loader.dataset)
    mae = total_abs_error / len(test_loader.dataset)
    return test_loss, mae



In [17]:
input_size = data["X_train_scaled"].shape[2]  # number of features
print(f"input_size (number of features): {input_size}")
sequence_length = data["X_train_scaled"].shape[1]  # sequence length
print(f"sequence_length: {sequence_length}")


input_size (number of features): 14
sequence_length: 30


In [20]:
# Train Transformer Model
print("\nTraining Transformer model...")
transformer_model = TransformerModel(
    input_size=input_size,
    hidden_size=64,
    num_layers=2,
    nhead=4,
    dropout=0.1
)
transformer_save_path = "best_transformer_model.pt"
transformer_model = train_model(
    transformer_model,
    train_loader,
    val_loader,
    num_epochs=50,
    lr=0.001,  # Starting with a slightly lower learning rate for transformer
    model_save_path=transformer_save_path,
)
transformer_test_loss, transformer_test_mae = evaluate_model(transformer_model, test_loader)
print(f"Transformer Model Test Loss: {transformer_test_loss:.4f}, Test MAE: {transformer_test_mae:.4f}")


Training Transformer model...
Training on device: cpu
Epoch 1/50, Train Loss: 141297.9295, Val Loss: 113889.3619
Saved best model at epoch 1 with val loss 113889.3619
Time taken for epoch 1: 17.03s
Epoch 2/50, Train Loss: 85426.5538, Val Loss: 62210.3192
Saved best model at epoch 2 with val loss 62210.3192
Time taken for epoch 2: 14.46s
Epoch 3/50, Train Loss: 53557.1100, Val Loss: 48336.1536
Saved best model at epoch 3 with val loss 48336.1536
Time taken for epoch 3: 13.12s
Epoch 4/50, Train Loss: 48592.0497, Val Loss: 47460.8774
Saved best model at epoch 4 with val loss 47460.8774
Time taken for epoch 4: 13.13s
Epoch 5/50, Train Loss: 48284.3025, Val Loss: 47225.2703
Saved best model at epoch 5 with val loss 47225.2703
Time taken for epoch 5: 13.89s
Epoch 6/50, Train Loss: 48174.7248, Val Loss: 47384.0258
Time taken for epoch 6: 14.71s
Epoch 7/50, Train Loss: 48182.4538, Val Loss: 47298.3495
Time taken for epoch 7: 15.13s
Epoch 8/50, Train Loss: 48029.8090, Val Loss: 46791.3602
Save